In [ ]:
# MTA SIRI API - Real-Time Bus Data

The MTA Bus Time API uses the SIRI (Service Interface for Real Time Information) standard to provide real-time bus locations, arrivals, and other transit data.

## Setup and API Key

1. Get your free API key from: https://bustime.mta.info/wiki/Developers/Index
2. Store it in your `.env` file as `MTA_API_KEY=your_key_here`

In [1]:
import requests
import json
import pandas as pd
from datetime import datetime
import os

# Load API key from environment
env = os.environ
MTA_API_KEY = env.get("MTA_API_KEY")

# Base URL for MTA Bus Time SIRI API
BASE_URL = "https://bustime.mta.info/api/siri/"

## 1. Vehicle Monitoring - Get Real-Time Bus Locations

This endpoint shows current positions of all buses on a specific route.

In [3]:
def get_vehicle_monitoring(route_id, api_key):
    """
    Get real-time vehicle positions for a specific bus route.
    
    Args:
        route_id: Bus route (e.g., 'M7', 'M15', 'B44')
        api_key: Your MTA API key
    
    Returns:
        DataFrame with bus locations and details
    """
    url = f"{BASE_URL}vehicle-monitoring.json"
    params = {
        'key': api_key,
        'LineRef': route_id,
        'VehicleMonitoringDetailLevel': 'calls'  # Include stop information
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # Parse the response
    vehicles = []
    vehicle_activities = data['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]['VehicleActivity']
    
    for vehicle in vehicle_activities:
        journey = vehicle['MonitoredVehicleJourney']
        
        vehicle_info = {
            'vehicle_id': journey.get('VehicleRef'),
            'route': journey.get('LineRef'),
            'direction': journey.get('DirectionRef'),
            'destination': journey.get('DestinationName'),
            'latitude': journey['VehicleLocation']['Latitude'],
            'longitude': journey['VehicleLocation']['Longitude'],
            'bearing': journey.get('Bearing'),
            'progress_rate': journey.get('ProgressRate'),  # e.g., 'normalProgress'
            'timestamp': vehicle['RecordedAtTime']
        }
        
        # Add next stop info if available
        if 'MonitoredCall' in journey:
            call = journey['MonitoredCall']
            vehicle_info['next_stop_name'] = call.get('StopPointName')
            vehicle_info['next_stop_id'] = call.get('StopPointRef')
            vehicle_info['distance_from_stop'] = call.get('Extensions', {}).get('Distances', {}).get('PresentableDistance')
        
        vehicles.append(vehicle_info)
    
    return pd.DataFrame(vehicles)

# Example: Get M7 bus locations
buses_df = get_vehicle_monitoring('M7', MTA_API_KEY)
print(f"Found {len(buses_df)} buses on route M7")
buses_df.head()

KeyError: 'VehicleActivity'

## 2. Stop Monitoring - Get Arrivals at a Specific Stop

This endpoint shows when buses will arrive at a particular stop.

In [ ]:
def get_stop_monitoring(stop_id, api_key):
    """
    Get upcoming bus arrivals at a specific stop.
    
    Args:
        stop_id: MTA stop ID (e.g., 'MTA_550005')
        api_key: Your MTA API key
    
    Returns:
        DataFrame with arrival predictions
    """
    url = f"{BASE_URL}stop-monitoring.json"
    params = {
        'key': api_key,
        'MonitoringRef': stop_id,
        'MaximumStopVisits': 10  # Number of upcoming arrivals
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # Parse arrivals
    arrivals = []
    stop_visits = data['Siri']['ServiceDelivery']['StopMonitoringDelivery'][0]['MonitoredStopVisit']
    
    for visit in stop_visits:
        journey = visit['MonitoredVehicleJourney']
        call = journey['MonitoredCall']
        
        arrival_info = {
            'route': journey.get('LineRef'),
            'destination': journey.get('DestinationName'),
            'vehicle_id': journey.get('VehicleRef'),
            'expected_arrival': call.get('ExpectedArrivalTime'),
            'aimed_arrival': call.get('AimedArrivalTime'),
            'stops_away': call.get('Extensions', {}).get('Distances', {}).get('StopsFromCall'),
            'distance': call.get('Extensions', {}).get('Distances', {}).get('PresentableDistance'),
            'vehicle_lat': journey['VehicleLocation']['Latitude'],
            'vehicle_lon': journey['VehicleLocation']['Longitude'],
        }
        arrivals.append(arrival_info)
    
    return pd.DataFrame(arrivals)

# Example: Get arrivals at a specific stop
# Note: You'll need to find a valid stop ID for your route
# arrivals_df = get_stop_monitoring('MTA_550005', MTA_API_KEY)
# arrivals_df.head()

## 3. Plot Real-Time Buses on a Map

In [2]:
import folium
from folium.plugins import MarkerCluster

def plot_realtime_buses(buses_df, route_id):
    """
    Plot real-time bus locations on an interactive map.
    """
    # Center map on Manhattan
    m = folium.Map(location=[40.7831, -73.9712], zoom_start=12, tiles="CartoDB positron")
    
    # Add each bus as a marker
    for _, bus in buses_df.iterrows():
        # Create popup with bus info
        popup_text = f"""
        <b>Route:</b> {bus['route']}<br>
        <b>Vehicle:</b> {bus['vehicle_id']}<br>
        <b>Destination:</b> {bus['destination']}<br>
        <b>Next Stop:</b> {bus.get('next_stop_name', 'N/A')}<br>
        <b>Distance:</b> {bus.get('distance_from_stop', 'N/A')}
        """
        
        # Add bus icon marker
        folium.Marker(
            location=[bus['latitude'], bus['longitude']],
            popup=folium.Popup(popup_text, max_width=300),
            icon=folium.Icon(color='blue', icon='bus', prefix='fa'),
            tooltip=f"Bus {bus['vehicle_id']}"
        ).add_to(m)
        
        # Add direction arrow if bearing is available
        if pd.notna(bus.get('bearing')):
            folium.RegularPolygonMarker(
                location=[bus['latitude'], bus['longitude']],
                fill_color='blue',
                number_of_sides=3,
                radius=8,
                rotation=bus['bearing']
            ).add_to(m)
    
    return m

# Plot the buses
if len(buses_df) > 0:
    m = plot_realtime_buses(buses_df, 'M7')
    m.save('realtime_buses_map.html')
    print(f"Map saved! Showing {len(buses_df)} buses")
    
    # Display in notebook
    from IPython.display import IFrame
    display(IFrame(src='realtime_buses_map.html', width=800, height=600))

NameError: name 'buses_df' is not defined

## 4. Additional Useful Functions

In [ ]:
# Get multiple routes at once
def get_multiple_routes(route_ids, api_key):
    """Get vehicle data for multiple routes."""
    all_buses = []
    for route in route_ids:
        try:
            buses = get_vehicle_monitoring(route, api_key)
            all_buses.append(buses)
            print(f"✓ {route}: {len(buses)} buses")
        except Exception as e:
            print(f"✗ {route}: Error - {e}")
    
    return pd.concat(all_buses, ignore_index=True) if all_buses else pd.DataFrame()

# Example: Get multiple Manhattan routes
manhattan_routes = ['M7', 'M15', 'M34A-SBS', 'M86-SBS']
# all_buses = get_multiple_routes(manhattan_routes, MTA_API_KEY)


# Continuous monitoring (run in a loop)
def monitor_route_continuously(route_id, api_key, duration_minutes=10, interval_seconds=30):
    """
    Monitor a route continuously and save snapshots.
    
    Args:
        route_id: Route to monitor
        api_key: MTA API key
        duration_minutes: How long to monitor
        interval_seconds: Time between API calls
    """
    import time
    
    snapshots = []
    start_time = datetime.now()
    end_time = start_time + pd.Timedelta(minutes=duration_minutes)
    
    print(f"Monitoring {route_id} for {duration_minutes} minutes...")
    
    while datetime.now() < end_time:
        try:
            buses = get_vehicle_monitoring(route_id, api_key)
            buses['snapshot_time'] = datetime.now()
            snapshots.append(buses)
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Captured {len(buses)} buses")
        except Exception as e:
            print(f"Error: {e}")
        
        time.sleep(interval_seconds)
    
    # Combine all snapshots
    all_data = pd.concat(snapshots, ignore_index=True)
    print(f"\n✅ Monitoring complete! Collected {len(all_data)} observations")
    return all_data

# Example usage (uncomment to run):
# historical_data = monitor_route_continuously('M7', MTA_API_KEY, duration_minutes=5, interval_seconds=30)

## Key Notes

**API Limits:**
- Free tier: 5,000 calls per day
- Rate limit: Recommended to wait 30 seconds between calls
- Data updates every 30 seconds

**Available Endpoints:**
1. `vehicle-monitoring.json` - Bus locations by route
2. `stop-monitoring.json` - Arrivals at a stop
3. `trip-updates.json` - Trip status updates (not covered here)

**Route Format:**
- Manhattan: M1, M7, M15, M34A-SBS, etc.
- Bronx: Bx1, Bx12, etc.
- Brooklyn: B35, B44-SBS, etc.
- Queens: Q10, Q44-SBS, etc.
- Staten Island: S40, S53, etc.

**Finding Stop IDs:**
- Check the MTA Bus Time website
- Or use the stop-monitoring endpoint with a known route to see stop IDs